# Analytical ELT/HARMONI pupil

Generate and inspect the ELT pupil without FITS files: 798 M1 segments, central obscuration, spiders and six projected M4 petals.

In [ ]:
import matplotlib.pyplot as plt
import torch

from fiatlux import ELTHarmoniPupil, FFTPropagator, Grid, PlaneWave, Spectrum
from fiatlux.core.spectrum import Band

torch.set_default_dtype(torch.float64)

grid = Grid(601, 601, 0.08, 0.08, dtype=torch.float64)
spectrum = Spectrum(
    magnitude=0, band=Band(1.65e-6, 0.0, 1.0), samples=1, dtype=torch.float64
)
pupil = ELTHarmoniPupil(
    grid,
    spider_width=0.5,
    spider_angles=(0.0, 60.0, 120.0),
    petal_gap=0.12,
    petal_rotation=30.0,
)
pupil.build(spectrum)

print(f"Active M1 segments: {pupil.number_of_segments}")
print(f"Sampled collecting area: {float(pupil.transmission.sum() * grid.dx * grid.dy):.2f} m²")
assert pupil.number_of_segments == 798

## Transmission and labels

`segment_index` and `petal_index` are `-1` outside the transmitted pupil. They can therefore be used directly to select a segment or an M4 petal for piston studies.

In [ ]:
extent = [float(grid.x.min()), float(grid.x.max()), float(grid.y.min()), float(grid.y.max())]
fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
images = [pupil.transmission, pupil.segment_index, pupil.petal_index]
titles = ["Transmission", "M1 segment index", "Projected M4 petal index"]
cmaps = ["gray", "turbo", "tab10"]
for axis, image, title, cmap in zip(axes, images, titles, cmaps):
    axis.imshow(image.cpu(), origin="lower", extent=extent, cmap=cmap)
    axis.set(title=title, xlabel="x [m]", ylabel="y [m]")
    axis.set_aspect("equal")
plt.show()

## Select petals and segments

In [ ]:
petal_areas = torch.stack([
    pupil.petal_mask(index).sum() * grid.dx * grid.dy for index in range(6)
])
coordinate = pupil.segment_coordinates[100]
selected_segment = pupil.segment_mask(coordinate)
print("Petal areas [m²]:", petal_areas.tolist())
print(f"Selected segment {coordinate}: {int(selected_segment.sum())} pixels")
assert torch.all(petal_areas > 0)

## Fraunhofer PSF

The analytical pupil is a normal Fiatlux mask and composes directly with the default Fraunhofer propagator.

In [ ]:
entrance = PlaneWave(spectrum).generate_field(grid)
focal = FFTPropagator(focal_length=1.0).apply(pupil.apply(entrance))
psf = focal.intensity()[0]
psf = psf / psf.max()
focal_extent = [
    float(focal.grid.x.min() * 1e6), float(focal.grid.x.max() * 1e6),
    float(focal.grid.y.min() * 1e6), float(focal.grid.y.max() * 1e6),
]
plt.figure(figsize=(6, 5))
plt.imshow(torch.log10(psf.clamp_min(1e-10)).cpu(), origin="lower", extent=focal_extent, cmap="inferno", vmin=-8, vmax=0)
plt.xlim(-0.5, 0.5)
plt.ylim(-0.5, 0.5)
plt.xlabel("x [µm]")
plt.ylabel("y [µm]")
plt.title("ELT/HARMONI Fraunhofer PSF [log10]")
plt.colorbar(label="log10 normalized intensity")
plt.show()